In [ ]:
import os
import sys
from pyspark.sql import SparkSession

# Correção de erro "Missing Python executable 'python3'"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Inicializa o Spark usando todos os núcleos disponíveis
spark = SparkSession.builder \
    .appName("ANAC_EDA_Bronze") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark

In [ ]:
# O caminho é relativo, pois o notebook está dentro da pasta /notebooks
caminho_bronze = "../data/bronze/dados_estatisticos.csv"

# Leitura exploratória
df_bronze = spark.read.csv(
    caminho_bronze,
    header=True,
    sep=";",
    encoding="ISO-8859-1",
    inferSchema=True
)

# Imprime a árvore de colunas e seus tipos identificadores
df_bronze.printSchema()

In [ ]:
import re
from unicodedata import normalize

# 1. Função de limpeza cirúrgica de strings
def padronizar_nome_coluna(nome):
    # Remove acentos e caracteres especiais
    nome_sem_acento = normalize('NFKD', nome).encode('ASCII', 'ignore').decode('ASCII')

    # Substitui parênteses, pontos e espaços por underline
    nome_sujo = re.sub(r'[^\w\s]', '_', nome_sem_acento)
    nome_espacos = re.sub(r'\s+', '_', nome_sujo)

    # Remove underlines duplicados e joga para minúsculo
    nome_limpo = re.sub(r'_+', '_', nome_espacos).strip('_').lower()
    return nome_limpo

# 2. Extrai os nomes atuais e aplica a função
colunas_originais = df_bronze.columns
colunas_padronizadas = [padronizar_nome_coluna(c) for c in colunas_originais]

# 3. Aplica a nova nomenclatura ao DataFrame de forma massiva
df_silver_raw = df_bronze.toDF(*colunas_padronizadas)

# Validação visual dos novos nomes e da anomalia de horas voadas
print("Novo Schema Padronizado")
print(df_silver_raw.columns)
print("\nInspecionando a anomalia na coluna 'horas_voadas':")
df_silver_raw.select("horas_voadas").filter(df_silver_raw.horas_voadas.isNotNull()).show(10)

In [ ]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.types import DoubleType

# Substitui a vírgula por ponto e altera o tipo de dado para numérico
df_silver_clean = df_silver_raw.withColumn(
    "horas_voadas",
    regexp_replace(col("horas_voadas"), ",", ".").cast(DoubleType())
)

# Validação estrita da conversão
print("Novo tipo da coluna:")
df_silver_clean.select("horas_voadas").printSchema()

print("\nDados convertidos (prontos para agregação matemática):")
df_silver_clean.select("horas_voadas").filter(col("horas_voadas").isNotNull()).show(10)